# Profilowanie regionalnego obciążenia sieci i awarii za pomocą PROC CHART

## Podsumowanie zarządcze

Analityk operacji sieciowych używa PROC CHART do sprofilowania syntetycznej próbki odczytów liczników obwodów zasilających w czterech regionach obsługi i czterech źródłach wytwarzania. Notatnik prezentuje wykresy słupkowe pionowe, słupkowe poziome, blokowe i kołowe, aby podsumować miks wytwarzania, porównać średnie i całkowite obciążenie według regionu, uwypuklić wieczorny szczyt zapotrzebowania według godziny oraz uszeregować minuty awarii według źródła — to rodzaj szybkiej eksploracji w formacie tekstowym, którą zespół energetyczno-usługowy przeprowadza przed przygotowaniem raportu graficznego. Krok DATA żąda 1200 wierszy; ten notatnik został wyrenderowany w trybie bez licencji, który ogranicza roboczy zbiór danych do pierwszych 100 odczytów, więc każdy wykres poniżej podsumowuje tę próbkę 100 wierszy.

## Źródła danych

| Zbiór danych | Wiersze | Opis |
|---------|------|-------------|
| `grid_ops` | 100 (próbka syntetyczna) | Jeden wiersz na odczyt licznika obwodu zasilającego w sieci regionalnej, generowany w miejscu za pomocą `call streaminit(20260531)` i `rand()`. Pętla kroku DATA żąda 1200 odczytów, ale tryb bez licencji ogranicza zapisany zbiór danych do 100 obserwacji, więc wykresy poniżej opisują te 100 wierszy. |

# Profilowanie regionalnego obciążenia sieci i awarii za pomocą PROC CHART

PROC CHART tworzy znakowe wykresy słupkowe, blokowe i kołowe bezpośrednio w listingu — bez potrzeby urządzenia ODS Graphics. Czyni to z niej szybkie narzędzie eksploracji wstępnej dla zespołu operacji sieciowych, który chce *zobaczyć* kształt swoich danych o obciążeniu i niezawodności przed zbudowaniem dopracowanych wizualizacji GCHART lub SGPLOT.

W tym notatniku:

1. Generujemy syntetyczny miesiąc odczytów liczników obwodów zasilających dla regionalnego operatora sieci.
2. Wykreślamy **miks wytwarzania** (udział odczytów według źródła).
3. Porównujemy **średnie i całkowite dostarczone obciążenie** w regionach obsługi.
4. Uwypuklamy **wieczorny szczyt zapotrzebowania** według godziny dnia.
5. Używamy **wykresu blokowego** do skrzyżowania regionu ze źródłem wytwarzania.
6. Szeregujemy **minuty awarii** według źródła, aby znaleźć najmniej niezawodne zasilanie.

Każda instrukcja i opcja poniżej to standardowa składnia SAS 9.4 PROC CHART.

## Krok 1 — Generowanie syntetycznych danych operacyjnych

Poniższy krok DATA fabrykuje odczyty liczników w pętli 1200 iteracji. Każdemu odczytowi przypisywany jest region obsługi, źródło wytwarzania (dominuje gaz, a resztę stanowią wiatr, energia słoneczna i woda) oraz godzina dnia. Obciążenie jest wyższe w wieczornym oknie 17:00–21:00 (i oznaczamy te odczyty jako szczytowe), a minuty awarii losujemy z rozkładu Poissona. `call streaminit` ustala ziarno losowe, dzięki czemu dane są odtwarzalne.

NOTE w logu pokazuje praktyczny rezultat: ten przebieg działa w trybie bez licencji, więc zapisany zbiór danych `grid_ops` jest ograniczony do pierwszych 100 z tych odczytów (100 wierszy, 7 kolumn). Każdy wykres, który następuje, podsumowuje tę próbkę 100 wierszy, a każdy log PROC CHART potwierdza, że odczytał 100 wierszy.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
DANE grid_ops;
    CALL streaminit(20260531);
    DŁUGOŚĆ region $12 source $9 peak_flag $3;
    TABLICA regions[4] $12 _temporary_
        ('North','South','East','West');
    POWTÓRZ meter_id = 1 TO 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        JEŚLI u < 0.42 WTEDY source = 'Gas';
        PRZECIWNIE JEŚLI u < 0.70 WTEDY source = 'Wind';
        PRZECIWNIE JEŚLI u < 0.88 WTEDY source = 'Solar';
        PRZECIWNIE source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        JEŚLI hour >= 17 AND hour <= 21 WTEDY POWTÓRZ;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        KONIEC;
        PRZECIWNIE POWTÓRZ;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        KONIEC;
        JEŚLI load_mwh < 0 WTEDY load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        WYJŚCIE;
    KONIEC;
    USUŃ r u BASE;
WYKONAJ;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Krok 2 — Miks wytwarzania

Jaki udział odczytów przypada na każde źródło wytwarzania? Wykres słupkowy pionowy z `TYPE=PERCENT` odpowiada na to bezpośrednio: wysokości słupków to procent wszystkich obserwacji przypadających na każdą kategorię źródła. Ponieważ `source` jest zmienną znakową, PROC CHART automatycznie traktuje jej wartości jako dyskretne kategorie.

In [2]:
PROCEDURA chart DANE=grid_ops;
    VBAR source / type=PERCENT;
WYKONAJ;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 3 — Średnie dostarczone obciążenie według regionu

Teraz przechodzimy od zliczania do podsumowywania pomiaru. Wskazanie `load_mwh` w `SUMVAR=` z `TYPE=MEAN` sprawia, że długość słupka to średnie obciążenie na region, a kolumny statystyk żądamy jawnie: `MEAN` drukuje średnią obok każdego słupka, a `CFREQ` dodaje kolumnę częstości skumulowanej. Region North ma najwyższe średnie dostarczone obciążenie (średnia około 28,0 MWh), zgodnie z regionalnym przesunięciem wbudowanym w dane, podczas gdy South jest najniższy (około 19,8 MWh).

Przekazujemy również `ASCENDING`, zamierzając uporządkować słupki od najniższej do najwyższej średniej. Zwróć uwagę w wyniku, że słupki **nie** są przeporządkowane — pojawiają się w kolejności kategorii (West, South, East, North), ze średnimi 24,2, 19,8, 21,7, 28,0, które nie biegną od najkrótszego do najdłuższego. Przeporządkowywanie słupków według statystyki wykresu nie jest jeszcze podłączone w tej implementacji PROC CHART, więc `ASCENDING`/`DESCENDING` są akceptowane, ale obecnie nie mają efektu; odczytaj ranking z wydrukowanej kolumny `Mean`.

In [3]:
PROCEDURA chart DANE=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
WYKONAJ;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 4 — Całkowite obciążenie według regionu

Tutaj `TYPE=SUM` sprawia, że każdy słupek to *całkowite* dostarczone obciążenie dla regionu, a nie średnia, więc wyższe słupki oznaczają regiony dostarczające najwięcej łącznej energii w próbkowanych odczytach. Region North ponownie prowadzi pod względem całkowitego dostarczonego obciążenia.

Instrukcja żąda również `SUBGROUP=peak_flag`, prosząc PROC CHART o podzielenie każdego słupka według tego, czy odczyt przypadł na wieczorne okno szczytowe. W SAS segmentuje to każdy słupek osobnym symbolem dla każdego poziomu podgrupy i drukuje legendę. Ta implementacja nie renderuje jeszcze segmentacji podgrup (udokumentowana luka funkcjonalna), więc słupki wychodzą jako pełne ciągi `*` bez podziału na segmenty — *sumy* słupków są poprawne, ale podział szczyt-vs-poza-szczytem nie jest tu pokazany. Aby zobaczyć, ile obciążenia przypada na godziny szczytu, użyj widoku według godziny dnia w Kroku 5.

In [4]:
PROCEDURA chart DANE=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
WYKONAJ;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 5 — Kształt obciążenia w ciągu dnia

`hour` jest ciągła, więc sami ustalamy podział na przedziały jawną listą `MIDPOINTS=` w centrach co 4 godziny (2, 6, 10, 14, 18, 22). Każdy słupek pokazuje średnie dostarczone obciążenie dla odczytów w pobliżu tej godziny. Słupek wyśrodkowany na 18 powinien się wyróżniać — to wieczorny szczyt zapotrzebowania, który wbudowaliśmy w dane.

In [5]:
PROCEDURA chart DANE=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
WYKONAJ;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Krok 6 — Region według źródła wytwarzania (wykres blokowy)

Wykres BLOCK renderuje małą tablicę dwukierunkową jako siatkę bloków. Z `GROUP=source` oraz `SUMVAR=load_mwh / TYPE=MEAN` wykres krzyżuje każdy region ze źródłem wytwarzania, które go obsługuje, a wysokość bloku jest proporcjonalna do średniego obciążenia — zwarty sposób na dostrzeżenie, które kombinacje region/źródło niosą najcięższe średnie obciążenie.

In [6]:
PROCEDURA chart DANE=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
WYKONAJ;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 7 — Miks wytwarzania jako wykres kołowy

Ta sama informacja o udziale źródeł z Kroku 2, narysowana jako wykres kołowy. PIE z `TYPE=PERCENT` skaluje każdy wycinek według jego procentu wszystkich odczytów i drukuje legendę procentów wycinków obok rysunku.

In [7]:
PROCEDURA chart DANE=grid_ops;
    PIE source / type=PERCENT;
WYKONAJ;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 8 — Minuty awarii według źródła

Na koniec szeregujemy niezawodność. `SUMVAR=outage_min` z `TYPE=SUM` sumuje minuty awarii na źródło. Przekazujemy `DESCENDING`, aby spróbować wypłynąć najgorzej działające źródło na górę, ale jak w Kroku 3 słupki nie są przeporządkowane — drukują się w kolejności kategorii (Hydro, Wind, Gas, Solar), a przeporządkowywanie słupków nie jest jeszcze zaimplementowane. Odczytaj ranking z wydrukowanej kolumny `Sum`: gaz odpowiada za najwięcej całkowitych minut awarii w tej próbce (122), znacznie wyprzedzając wiatr (64), energię słoneczną (43) i wodę (38). To odzwierciedla miks wytwarzania — gaz obsługuje najwięcej odczytów, więc gromadzi najwięcej minut awarii ogółem.

In [8]:
PROCEDURA chart DANE=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum MALEJĄCO;
WYKONAJ;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interpretacja wyników

Odczytanie wykresów razem daje zespołowi operacyjnemu szybki obraz sytuacji (na próbce 100 wierszy uchwyconej w tym przebiegu):

- **Miks wytwarzania (Kroki 2 i 7).** Gaz ma największy udział odczytów (około 45%), z wiatrem na drugim miejscu (około 28%) oraz wodą i energią słoneczną w tyle (około 14% i 13%) — wykres słupkowy pionowy i kołowy opowiadają tę samą historię na dwa sposoby, co jest przydatnym testem spójności.
- **Obciążenie według regionu (Kroki 3 i 4).** HBAR średniego obciążenia pokazuje North z najwyższym średnim dostarczonym obciążeniem (średnia ~28 MWh) i South z najniższym (~20 MWh), zgodnie z regionalnym przesunięciem wbudowanym w dane. Wykres SUM potwierdza, że North dostarcza również najwięcej całkowitej energii.
- **Kształt dziennego obciążenia (Krok 5).** Słupek punktu środkowego wyśrodkowany na godzinie 18 jest wyraźnie najwyższy, potwierdzając szczyt zapotrzebowania 17:00–21:00, który wbudowaliśmy w dane — dokładnie tam, gdzie przedsiębiorstwo skupiłoby planowanie reakcji na zapotrzebowanie i mocy.
- **Niezawodność (Krok 8).** Sumowanie minut awarii według źródła uwypukla gaz jako największego kontrybutora przestojów w tej próbce (122 minuty), naturalny kolejny cel przeglądu konserwacji — choć odzwierciedla to głównie fakt, że gaz obsługuje najwięcej odczytów.

Dwie opcje wyświetlania użyte tutaj — przeporządkowywanie słupków `ASCENDING`/`DESCENDING` (Kroki 3 i 8) oraz segmentacja słupków `SUBGROUP=` (Krok 4) — są akceptowane przez parser, ale nie są jeszcze renderowane przez tę implementację PROC CHART, więc rankingi i podziały odczytuje się z wydrukowanych kolumn statystyk, a nie z kolejności czy cieniowania słupków.

PROC CHART daje wyjście wyłącznie znakowe, więc dla wizualizacji gotowych do prezentacji zarządowi zespół przeniósłby te same widoki do PROC GCHART lub PROC SGPLOT. Ale jako pierwsze przejście bez konfiguracji przez świeży ekstrakt, te wykresy odpowiadają na pytania operacyjne — miks, wielkość i czas — w kilka sekund.